# **Table of Contents**
* [Your Notebook](#your-notebook)
  * [Table of Contents](#table-of-contents)
* [TOC Generator](#toc-generator)


imports

In [1]:
import pandas as pd

#### Clean Poverty Data

In [2]:
df_poverty = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_poverty_by_tract_s1701.csv')

df_poverty.columns

Index(['GEO_ID', 'NAME', 'S1701_C01_001E', 'S1701_C01_001M', 'S1701_C01_002E',
       'S1701_C01_002M', 'S1701_C01_003E', 'S1701_C01_003M', 'S1701_C01_004E',
       'S1701_C01_004M',
       ...
       'S1701_C03_058M', 'S1701_C03_059E', 'S1701_C03_059M', 'S1701_C03_060E',
       'S1701_C03_060M', 'S1701_C03_061E', 'S1701_C03_061M', 'S1701_C03_062E',
       'S1701_C03_062M', 'Unnamed: 374'],
      dtype='object', length=375)

In [3]:
df_poverty = df_poverty[["GEO_ID", "NAME", "S1701_C01_001E", "S1701_C02_001E", "S1701_C03_001E"]]

# The first row labels are redundant
df_poverty.drop(index = 0, inplace = True)

# Rename to useful column names
df_poverty = df_poverty.rename(columns={
    "NAME" : "TRACT",
    "S1701_C01_001E" : "POPULATION",
    "S1701_C02_001E" : "BELOW_POVERTY",
    "S1701_C03_001E" : "POVERTY_RATE"
    })

# Keep only tract # in string
df_poverty['TRACT'] = df_poverty['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later
df_poverty['TRACT'] = df_poverty['TRACT'].astype(float)

df_poverty.head()

,GEO_ID,TRACT,POPULATION,BELOW_POVERTY,POVERTY_RATE
1,1400000US21111000201,2.01,1163,398,34.2
2,1400000US21111000202,2.02,706,370,52.4
3,1400000US21111000300,3.00,2155,585,27.1
4,1400000US21111000400,4.00,4644,955,20.6
5,1400000US21111000600,6.00,1252,328,26.2


In [4]:
df_poverty.to_csv("../Curtis/data/processed/poverty_rate_clean.csv", index = False)

#### Clean Income Data

In [5]:
df_income = pd.read_csv('../Curtis/data/raw/acs_2024_jefferson_household_income_by_tract_b19013.csv')

df_income.head()

,Label (Grouping),Median household income in the past 12 months (in 2024 inflation-adjusted dollars)
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [6]:
# Rename columns
df_income = df_income.rename(columns={
    "Label (Grouping)" : "TRACT",
    "Median household income in the past 12 months (in 2024 inflation-adjusted dollars)" : "MEDIAN_ESTIMATED_HOUSEHOLD_INCOME"
    })

df_income.head()

,TRACT,MEDIAN_ESTIMATED_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,NaN
1,Estimate,"35,825"
2,Margin of Error,"±11,473"
3,Census Tract 2.02; Jefferson County; Kentucky,NaN
4,Estimate,"17,039"


In [7]:
# Shift values up one row so Estimate will be on the same row as tract
df_income['MEDIAN_ESTIMATED_HOUSEHOLD_INCOME'] = df_income['MEDIAN_ESTIMATED_HOUSEHOLD_INCOME'].shift(-1)

df_income.head()

,TRACT,MEDIAN_ESTIMATED_HOUSEHOLD_INCOME
0,Census Tract 2.01; Jefferson County; Kentucky,"35,825"
1,Estimate,"±11,473"
2,Margin of Error,NaN
3,Census Tract 2.02; Jefferson County; Kentucky,"17,039"
4,Estimate,"±6,250"


In [8]:
# Simplify the layout to be Tract #: Estimate value and delete Margin of Error rows
df_income = df_income.drop(df_income[df_income['TRACT'].str.contains('Estimate|Margin of Error', na=False)].index)

# Keep only tract # in string
df_income['TRACT'] = df_income['TRACT'].str.extract(r'Census Tract ([\d.]+)')

# Treat Tract # as a float for merging later
df_income['TRACT'] = df_income['TRACT'].astype(float)

# Make Tract the new index
df_income = df_income.set_index('TRACT')

df_income.head()

,MEDIAN_ESTIMATED_HOUSEHOLD_INCOME
TRACT,
2.01,"35,825"
2.02,"17,039"
3.00,"34,764"
4.00,"41,713"
6.00,"30,734"


In [9]:
df_income.to_csv("../Curtis/data/processed/income_clean.csv", index = True)

# TOC Generator 

Do not delete code. It will read your markdown and generate a TOC we can put in the notebook for easy navigation. 

In [10]:
import json
import os


def generate_toc_from_notebook(notebook_path):
    """
    Parses a local .ipynb file and generates Markdown for a Table of Contents.
    """
    if not os.path.isfile(notebook_path):
        print(f"❌ Error: File not found at '{notebook_path}'")
        return

    with open(notebook_path, 'r', encoding='utf-8') as f:
        notebook = json.load(f)

    toc_markdown = "# **Table of Contents**\n"
    for cell in notebook.get('cells', []):
        if cell.get('cell_type') == 'markdown':
            for line in cell.get('source', []):
                if line.strip().startswith('#'):
                    level = line.count('#')
                    title = line.strip('#').strip()
                    link = title.lower().replace(' ', '-').strip('-.()')
                    indent = '  ' * (level - 1)
                    toc_markdown += f"{indent}* [{title}](#{link})\n"

    print("\n--- ✅ Copy the Markdown below and paste"
          "it into a new markdown cell ---\n")
    print(toc_markdown)


if __name__ == "__main__":
    # Example usage
    notebook_path = 'curtis.ipynb'  # Replace with your notebook path
    generate_toc_from_notebook(notebook_path)


--- ✅ Copy the Markdown below and pasteit into a new markdown cell ---

# **Table of Contents**
* [**Table of Contents**](#**table-of-contents**)
      * [Clean Poverty Data](#clean-poverty-data)
      * [Clean Income Data](#clean-income-data)
* [TOC Generator](#toc-generator)

